# 🎙️ LifeMate v2.0 — High-Performance SFT Pipeline
### Zalo AI Challenge 2026

This notebook implements the complete **v2.0 Top-Tier Pipeline**:
1. **Audio Tai Nghe**: NVIDIA Parakeet FastConformer-CTC (`nvidia/parakeet-ctc-0.6b-vi`) with PhoWhisper fallback.
2. **Agent Não Bộ**: Supervised Fine-Tuning (SFT LoRA) on `Qwen/Qwen2.5-3B-Instruct` across 1,545 multi-turn trajectories.
3. **Auto-Merge**: Merges LoRA weights into a standalone offline model.
4. **Inference & Validation**: Runs interactive `zaic_tools` agent on `public_test` and validates schema compliance.

In [ ]:
# 1. GPU Check
!nvidia-smi

In [ ]:
# 2. Install Core Dependencies
!pip install -q soundfile accelerate peft trl datasets

# Install NVIDIA NeMo for Parakeet ASR
!pip install -q nemo_toolkit[asr]

In [ ]:
# 3. Clone Repository
import os
REPO_URL = "https://github.com/Quandinh030106/LifeMate-Voice2Action-Agent.git"
if not os.path.exists("/kaggle/working/LifeMate-Voice2Action-Agent"):
    !git clone $REPO_URL /kaggle/working/LifeMate-Voice2Action-Agent
else:
    !cd /kaggle/working/LifeMate-Voice2Action-Agent && git pull origin main

%cd /kaggle/working/LifeMate-Voice2Action-Agent

In [ ]:
# 4. Locate Dataset Directory
import glob, os
datasets = glob.glob("/kaggle/input/**/public_test.jsonl", recursive=True)
if datasets:
    DATA_DIR = os.path.dirname(os.path.dirname(datasets[0]))
    print("Found Data Directory:", DATA_DIR)
else:
    DATA_DIR = "/kaggle/input/lifemate-data/train+public_test/train+public_test"
    print("Defaulting to:", DATA_DIR)

In [ ]:
# 5. Step 1: Prepare SFT Training Data from train.jsonl (1545 multi-turn samples)
!python -u scripts/prepare_sft_data.py \
    --data_dir "$DATA_DIR" \
    --output_dir data/sft

In [ ]:
# 6. Step 2: Fine-Tune Qwen2.5-3B-Instruct via LoRA and Merge (~20-25 mins on GPU T4)
import subprocess, sys

# Set SKIP_TRAIN = True only if you already trained and saved the model previously
SKIP_TRAIN = False

if not SKIP_TRAIN:
    cmd = [
        sys.executable, "-u", "train_sft.py",
        "--model_name_or_path", "Qwen/Qwen2.5-3B-Instruct",
        "--train_file", "data/sft/sft_train.jsonl",
        "--val_file", "data/sft/sft_val.jsonl",
        "--output_dir", "/kaggle/working/qwen-lora",
        "--merged_dir", "/kaggle/working/lifemate-qwen-sft",
        "--epochs", "3",
        "--batch_size", "2",
        "--grad_accum", "4",
        "--lr", "2e-4"
    ]
    print("Executing:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skipping training, using existing model at /kaggle/working/lifemate-qwen-sft")

In [ ]:
# 7. Step 3: Run Full Inference on public_test with NVIDIA Parakeet + Fine-Tuned Agent (~15-18 mins)
!python -u inference.py \
    --data_dir "$DATA_DIR" \
    --split public_test \
    --asr_engine auto \
    --llm_model /kaggle/working/lifemate-qwen-sft \
    --output /kaggle/working/submission.jsonl

In [ ]:
# 8. Step 4: Validate Output Schema with Official zaic_tools Validator
!PYTHONPATH="$DATA_DIR/public_test:$PYTHONPATH" python -m zaic_tools \
    --split public_test \
    --root "$DATA_DIR" \
    --validate /kaggle/working/submission.jsonl

In [ ]:
# 9. Step 5: Check submission line count and preview
!wc -l /kaggle/working/submission.jsonl
!head -n 5 /kaggle/working/submission.jsonl

In [ ]:
# 10. Step 6: Backup LoRA Adapter for Future Runs (~50MB only)
import os, shutil
adapter_dir = "/kaggle/working/qwen-lora"
if os.path.exists(adapter_dir):
    shutil.make_archive("/kaggle/working/qwen_lora_adapter", "zip", adapter_dir)
    print("LoRA adapter archived successfully at /kaggle/working/qwen_lora_adapter.zip")
    !ls -lh /kaggle/working/qwen_lora_adapter.zip
else:
    print("No adapter directory found to archive.")